In [1]:
import pandas as pd
import sys
import numpy as np
import warnings
import os

sys.path.append("/Users/ejowik001/Desktop/Github/InsightsNow/app/analytical-backend/src/maynard/dependencies")

In [2]:
from estimation import (
    cast_to_base_unit,
    estimate_automl,
    calculate_contributions,
    calculate_conf_bounds,
)

from plots import plot_prediction
from retransform_prediction import retransform_
from retransform_data import retransform_data
from tools import cast_spec_to_dict, _convert_to_datetime

In [3]:
from dateutil.relativedelta import relativedelta

In [4]:
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri


# Activate automatic conversion between pandas and R dataframes
pandas2ri.activate()

# Import the R package
mbreaks = importr('mbreaks')

In [5]:
from sklearn.metrics import r2_score

In [6]:
EPSILON = 1e-10

def rrse(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray=None):
    """ Root Relative Squared Error """
    return np.sqrt(
        np.sum(np.square(actual - predicted))
        / np.sum(np.square(actual - benchmark))
    )

def _error(actual: np.ndarray, predicted: np.ndarray):
    """ Simple error """
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """
    Percentage error

    Note: result is NOT multiplied by 100
    """
    return _error(actual, predicted) / (actual + EPSILON)

def mape(actual: np.ndarray, predicted: np.ndarray):
    """
    Mean Absolute Percentage Error

    Note: result is NOT multiplied by 100
    """
    return np.mean(np.abs(_percentage_error(actual, predicted)))

def mse(actual: np.ndarray, predicted: np.ndarray):
    """ Mean Squared Error """
    return np.mean(np.square(_error(actual, predicted)))


def rmse(actual: np.ndarray, predicted: np.ndarray):
    """ Root Mean Squared Error """
    return np.sqrt(mse(actual, predicted))


In [7]:
def assign_weights(s):
    if (s['directional_accuracy'] == -1) and (s['within_cbounds'] == -1):
        return 2
    elif (s['directional_accuracy'] == -1) and (s['within_cbounds'] == 1):
        return 1.75
    elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == -1):
        return 1.25
    elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == 1):
        return 1
    else: return np.infty

In [8]:
confidence_bounds_func = lambda row: row['lower']<=row['y_pred']<=row['upper']

In [9]:
def wdmpe(predicted, actual):
    actual_diff = actual.sort_index().diff()
    actual_signs = np.sign(actual_diff)
    predicted_diff = predicted.sort_index().diff()
    predicted_signs = np.sign(predicted_diff)

    resid = predicted-actual

    dir_acc = list(actual_signs * predicted_signs)

    resid_mean = resid.expanding(1).mean()
    resid_std = resid.expanding(2).std().fillna(0)

    lower = actual-resid_std
    upper = actual+resid_std

    df = pd.DataFrame({
        "directional_accuracy": dir_acc,
        "lower": lower,
        "upper": upper,
        "y_pred": predicted
    }).iloc[1:, :]
    df['within_cbounds'] = df.apply(confidence_bounds_func, axis=1).astype(int).replace({0: -1})
    df['percentage_error'] = resid / actual

    df['weights'] = df.apply(assign_weights, axis=1)
    df["weighted_percentage_error"] = df['weights'] * df['percentage_error']
    return df["weighted_percentage_error"].mean()


In [10]:
series_name = "PCEC96"
reference_date = "2023-12-01"
n_periods = 72

In [11]:
ds = pd.read_parquet("./app/analytical-backend/data/04_feature/selected_series.parquet")

In [12]:
spec = pd.read_csv("./app/analytical-backend/data/02_intermediate/variable.csv")
ds_base = pd.read_parquet("./app/analytical-backend/data/02_intermediate/non_transformed_data.parquet")

In [13]:
# Example usage
model_result = estimate_automl(
    ds=ds,
    ds_base=ds_base,
    spec=spec,
    ref_date_col="ReferenceDate",
    series_name=series_name,
    reference_date=reference_date,
    n_periods=n_periods,
)

In [14]:


# reference_date = pd.to_datetime(parameters["ref_date"]).date()
reference_date = pd.to_datetime(reference_date)
lag_date = reference_date - relativedelta(months=1)
# lag = model_result["pred_"]["backcast"].loc[(reference_date-relativedelta(months=1)).strftime("%Y-%m-%d")]
pred = model_result["pred_"]["forecast"]
coef_ = model_result["coef_"]
values = model_result["values"]

# Print the best model's details
print("============ Model Details ============")
print(f"Model                     : {model_result['best_model']}")
print(f"Reference Date            : {reference_date}")
print(f"Forecast                  : {pred:.4f}")
print(f"R-Squared (R²)            : {model_result['r_squared']:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {model_result['mape']:.2f}%")
print(f"Root Mean Square Error (RMSE) : {model_result['rmse']:.4f}")
print("\n")

# print(calculate_contributions(coef_, pred, lag, values))

formula = spec.loc[spec["seriesid"] == series_name][
    "transformation"
].item()
unit = spec.loc[spec["seriesid"] == series_name]["units"].item()
dt = model_result["pred_"]["backcast"].index

pred = model_result["pred_"]["backcast"]
actual = model_result["actual"].loc[dt]
bounds = calculate_conf_bounds(pred, actual)

bounds_level = {}
for key, value in bounds.items():
    data, dt_, _ = cast_to_base_unit(
        ds_base, spec, series_name, value, dtype="pred"
    )
    tmp = pd.Series(data.reshape(1, -1)[0], index=dt_)
    bounds_level[key] = tmp.loc[dt]

# plot_prediction(
#     dt=dt,
#     y_pred=pred,
#     y_actual=actual,
#     mode="lines+markers",
#     lower1=bounds["L1"],
#     upper1=bounds["U1"],
#     lower2=bounds["L2"],
#     upper2=bounds["U2"],
#     title=f'Series: {series_name}, Reference Date: {reference_date}, Unit: {unit} {formula}',
#     # plt_out_path=os.path.join(parameters["fig_out_dir"], f'{model_result["best_model"]}_{datetime.now().strftime("%Y%m%d%H%M%S")}_pva.png')
# )

transf_pred = pd.concat(
    [
        model_result["pred_"]["backcast"],
        pd.Series(
            model_result["pred_"]["forecast"],
            index=[pd.to_datetime(model_result["pred_"]["reference_date"])],
        ),
    ]
)
transf_actual = model_result["actual"]

# Retransform forecast
Rhat, Time, cutoff_date = cast_to_base_unit(
    ds_base, spec, series_name, transf_pred, dtype="pred"
)
R, _, _ = cast_to_base_unit(
    ds_base, spec, series_name, transf_actual, dtype="actual"
)

# TBC
header = [series_name]
Rhat_df = pd.DataFrame(Rhat, columns=header, index=Time)
R_df = pd.DataFrame(R, columns=header, index=Time)

retr_forecast = Rhat_df.loc[reference_date.date()].item()
retr_actual = R_df.loc[reference_date.date()].item()
retr_lag = R_df.loc[lag_date.date()].item()

contributions = calculate_contributions(coef_, retr_forecast, retr_lag, values)

###################     HERE     ###################
start_date, end_date = model_result["pred_"]["backcast"].index.min(), reference_date
pred_act =  pd.concat([
    Rhat_df.loc[start_date.date():reference_date.date()].rename(columns={series_name: "Predicted"}),
    R_df.loc[start_date.date():reference_date.date()].rename(columns={series_name: "VariableValue"}),
    Rhat_df.shift().loc[start_date.date():reference_date.date()].rename(columns={series_name: "Lag"})
], axis=1)

print("\n============ Forecast vs Actual ============")
print(f"Reference Date            : {reference_date}")
print(f"Retransformed Forecast    : {retr_forecast:,.2f}")
print(f"Actual Release            : {retr_actual:,.2f}")
# print(
#     f"Percentage Error (Level)  : {(retr_forecast - retr_actual) / retr_actual:.2%}"
# )
print(f"R-Squared (R²)            : {r2_score(y_true=pred_act["VariableValue"], y_pred=pred_act["Predicted"]):.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape(actual=pred_act["VariableValue"], predicted=pred_act["Predicted"]):.2f}%")
print(f"Root Mean Square Error (RMSE) : {mse(actual=pred_act["VariableValue"], predicted=pred_act["Predicted"]):.4f}")

pred_act["Lag(Actual Change (MoM))"] = (pred_act["VariableValue"] - pred_act["Lag"]).shift()
pred_act["Predicted - Lag"] = pred_act["Predicted"] - pred_act["Lag"]
df = pred_act.reset_index().rename(columns={"index": "ReferenceDate"}).dropna(subset=["Lag"])

r_df = pandas2ri.py2rpy(df)

# result = mbreaks.mdl(y_name="Lag", data=r_df)
# # List available keys
# print(result.names)
# # Example: Access BIC values
# print(result.rx2('SEQ'))

strucchange = importr('strucchange')

ro.globalenv['y'] = ro.FloatVector(df["Lag"])
ro.globalenv['reference_date'] = ro.StrVector(df["ReferenceDate"].astype(str))

ro.r('''
    data <- data.frame(y = y, reference_date = reference_date)
    bp_model <- breakpoints(y ~ 1, data = data)
    bp_dates <- data$reference_date[bp_model$breakpoints]
''')

# Extract breakpoints
breakpoints = ro.r('bp_model$breakpoints')
print("\n")
print("Detected Breakpoints at:", list(breakpoints))

bp_dates = ro.r('as.character(bp_dates)')
bp_dates = list(bp_dates)

print("Structural breaks detected at:", bp_dates)

pred_act.index = pd.to_datetime(pred_act.index)
bp_dates = pd.to_datetime(bp_dates)

baseline = (pred_act["Lag(Actual Change (MoM))"] / pred_act["Lag"]).abs().rolling(6).quantile(0.9)  # adaptively high
ratio = ((pred_act["Predicted - Lag"]/pred_act["Lag"]).abs() > np.maximum(baseline, 0.015))

dynamic_windows = []
start, end = np.nan, np.nan

for i in range(len(bp_dates)):

    start = bp_dates[i]
    try:
        end = bp_dates[i+1]
    except IndexError:
        end = ratio.index.max()

    window = (ratio > baseline).loc[start:end]

    if window.any():
        end_date = window.loc[window].index.max()
        dynamic_windows.append((start, end_date))

# Nowcasts adjustment
pred_act = pred_act.reset_index().rename(columns={"index": "ReferenceDate"})
pred_act["in_dynamic_window"] = False

# Check for each window
for start, end in dynamic_windows:
    pred_act["in_dynamic_window"] |= pred_act["ReferenceDate"].between(start, end)

# Flag preview
# display(pred_act)
adjusted_nowcast = pred_act["Lag"]

pred_act["Nowcast"] = np.where(
    pred_act["in_dynamic_window"],
    adjusted_nowcast,
    pred_act["Predicted"]
    )

# Log.INFO
formatted_windows = [
    (start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
    for start, end in dynamic_windows
]

print("Nowcasts adjusted for periods flagged by the Bai-Perron structural break test:", formatted_windows)
print("\n")

print("=============== Summary ===============")
print(f"R-Squared (R²)            : {r2_score(y_true=pred_act["VariableValue"], y_pred=pred_act["Nowcast"]):.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape(actual=pred_act["VariableValue"], predicted=pred_act["Nowcast"]):.2f}%")
print(f"Root Mean Square Error (RMSE) : {mse(actual=pred_act["VariableValue"], predicted=pred_act["Nowcast"]):.4f}")


plot_prediction(
    dt=dt,
    y_pred=Rhat_df.loc[dt][series_name],
    y_actual=R_df.loc[dt][series_name],
    lower1=bounds_level["L1"],
    upper1=bounds_level["U1"],
    lower2=bounds_level["L2"],
    upper2=bounds_level["U2"],
    mode="lines+markers",
    title=f'Series: {series_name}, Reference Date: {reference_date}, Unit: {unit}',
    # plt_out_path=os.path.join(parameters["fig_out_dir"], f'{model_result["best_model"]}_{datetime.now().strftime("%Y%m%d%H%M%S")}_bpva.png')
)
plot_prediction(
    dt=dt,
    y_pred=pred_act.set_index("ReferenceDate").loc[dt]["Nowcast"],
    y_actual=R_df.loc[dt][series_name],
    lower1=bounds_level["L1"],
    upper1=bounds_level["U1"],
    lower2=bounds_level["L2"],
    upper2=bounds_level["U2"],
    mode="lines+markers",
    title=f'Series: {series_name}, Reference Date: {reference_date}, Unit: {unit}',
    # plt_out_path=os.path.join(parameters["fig_out_dir"], f'{model_result["best_model"]}_{datetime.now().strftime("%Y%m%d%H%M%S")}_bpva.png')
)

============ Model Details ============
Model                     : Ridge
Reference Date            : 2023-12-01 00:00:00
Forecast                  : 0.4325
R-Squared (R²)            : 0.9031
Mean Absolute Percentage Error (MAPE): 3.76%
Root Mean Square Error (RMSE) : 1.5478



============ Forecast vs Actual ============
Reference Date            : 2023-12-01 00:00:00
Retransformed Forecast    : 15,624.79
Actual Release            : 15,649.60
R-Squared (R²)            : 0.9875
Mean Absolute Percentage Error (MAPE): 0.00%
Root Mean Square Error (RMSE) : 7636.6564


Detected Breakpoints at: [39.0, 56.0]
Structural breaks detected at: ['2021-03-01', '2022-08-01']
Nowcasts adjusted for periods flagged by the Bai-Perron structural break test: [('2021-03-01', '2021-03-01'), ('2022-08-01', '2023-01-01')]


=============== Summary ===============
R-Squared (R²)            : 0.9793
Mean Absolute Percentage Error (MAPE): 0.00%
Root Mean Square Error (RMSE) : 12602.1514


In [15]:
# out_dir = "/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/07_model_output/"

# # Save everything to an Excel file with multiple sheets
# excel_file = os.path.join(out_dir, f'ml.xlsx')

# with pd.ExcelWriter(excel_file, engine='xlsxwriter') as writer:
#     # Save model details as a dataframe
#     sheet1 = pd.DataFrame.from_dict({"Model Name": model_result["best_model"], "R-Squared": model_result["r_squared"], "MAPE": model_result["mape"], "RMSE": model_result["rmse"]}, orient="index", columns=["Value"]).reset_index().rename(columns={"index": "Banner"})
#     sheet1.to_excel(writer, sheet_name="Model Details", index=False)
#     # Save contributions
#     contributions.to_excel(writer, sheet_name="Contributions")

#     # Save forecast and actual values
#     R_df = R_df.rename(columns={series_name: "Actual"})
#     Rhat_df = Rhat_df.rename(columns={series_name: "Predicted"})
#     sheet3 = pd.merge(R_df, Rhat_df, how="outer", left_index=True, right_index=True).reset_index().rename(columns={"index": "Reference Date"})
#     sheet3.to_excel(writer, sheet_name="Forecast vs Actual", index=False)

# print(f"Results saved to {excel_file}")